# 🐉 Imagen → 3D con Hunyuan3D-2 (gratis) — con todos los parches

Convierte **una imagen de personaje (SIN FONDO)** en un **modelo 3D** (`.glb`) en la GPU gratis de Colab.
Trae cada parche real que hizo falta para que funcione:

| Problema real | Parche |
|---|---|
| `AttributeError … _blas_supports_fpe` al importar | **Celda 1B**: `numpy>=2.1` + **Reiniciar sesión** |
| Warnings `numba / cuml / cudf incompatible` | inofensivos, se ignoran |
| El widget de subir archivos no anda (sobre todo en **celular**) | **Celda 2**: sube por widget **o** por el panel de Archivos 📁 |
| `Model path not exists` con el **mini** | **Celda 3**: modelo **completo** `tencent/Hunyuan3D-2` |
| `CUDA out of memory` | **Celda 3B**: `octree_resolution=192` |

> ⚠️ **Regla de oro:** subí la imagen **YA SIN FONDO** (PNG transparente, recortada). Con fondo, el modelo lo interpreta como cuerpo y sale deforme.

## Orden
1. **GPU T4**: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.
2. **Celda 1** (instalar) → **Celda 1B** (numpy) → **Reiniciar sesión**.
3. **Celda 2** (subir la imagen sin fondo).
4. **Celda 3** (generar). Si da *out of memory* → **Celda 3B**.
5. **Celda 4** (descargar `.glb`).

## Celda 1 — Instalar Hunyuan3D-2
Tarda ~5–8 min la primera vez. Solo lo necesario para la **forma** (no compila los módulos CUDA de textura, que fallan seguido en Colab).

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/Hunyuan3D-2'):
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
os.chdir('/content/Hunyuan3D-2')

# Parche para la TEXTURA (Celda 3C): el modelo de pintura 'hunyuanpaint' es codigo
# custom y diffusers exige trust_remote_code=True, que el codigo de Hunyuan no pasa.
# Lo agregamos directo al archivo (asi no falla al texturizar).
!sed -i 's/torch_dtype=torch\.float16)/torch_dtype=torch.float16, trust_remote_code=True)/' hy3dgen/texgen/utils/multiview_utils.py

# Dependencias de la generacion de FORMA
!pip install -q ninja
!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub
!pip install -q -e . 2>&1 | tail -3          # instala hy3dgen (ignora warnings de version)

import torch
print('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('✅ Celda 1 lista. Ahora corré la Celda 1B.')

## Celda 1B — Parche `numpy` → **REINICIAR SESIÓN**
Sin esto, la Celda 3 tira `AttributeError: _blas_supports_fpe` al importar `trimesh`/`hy3dgen` (a `scipy` le falta ese símbolo en numpy < 2.1).
Los warnings de `numba / cuml / cudf incompatible` que aparezcan son **inofensivos** (son de RAPIDS, no los usamos).

In [ ]:
!pip install -q -U "numpy>=2.1"
print('\n⚠️  AHORA hacé:  Entorno de ejecución → Reiniciar sesión')
print('   (obligatorio para que tome el numpy nuevo). Después seguí con la Celda 2.')

## Celda 2 — Subir tu imagen (SIN FONDO)
Dos formas (usá la que te funcione, **en celular la opción B es más confiable**):
- **A)** Corré esta celda y usá el botón *Elegir archivos*.
- **B)** Abrí el panel **Archivos** 📁 (ícono de carpeta a la izquierda), arrastrá/subí tu PNG, y corré esta celda: detecta sola la imagen más reciente en `/content`.

In [ ]:
import os, glob
from PIL import Image

IMG = None
# Opcion A: widget de subida (si anda). files.upload() guarda el archivo en la carpeta actual.
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget de subida no anduvo (', e, ') -> uso la opción B.')

# Opcion B: buscar la imagen mas reciente subida por el panel Archivos 📁 (queda en /content)
if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré imagen. Subila por el botón de arriba o por el panel Archivos 📁 y volvé a correr esta celda.'
im = Image.open(IMG)
transp = (im.mode == 'RGBA')
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Transparente (sin fondo).' if transp else '⚠️ NO es transparente. Para que no salga deforme, subí un PNG recortado sin fondo.')

## Celda 3 — Generar el modelo 3D (forma)
Modelo **completo** `tencent/Hunyuan3D-2` (el `mini` da `Model path not exists`).
La **primera vez** baja ~10 GB de pesos → paciencia (barras de descarga). Después genera la malla (~30–60 s en la T4). Exporta a `/content/hunyuan_mesh.glb`.

In [ ]:
import os, torch
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

print('Cargando el modelo (la 1a vez baja ~10 GB)...')
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
print('✅ Modelo cargado.')

img = Image.open(IMG).convert('RGBA')   # conserva la transparencia (imagen sin fondo)
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=256,
            generator=torch.manual_seed(0))[0]

OUT = '/content/hunyuan_mesh.glb'
os.makedirs(os.path.dirname(OUT), exist_ok=True)   # por las dudas, la carpeta destino existe
mesh.export(OUT)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB')
      if os.path.exists(OUT) else '❌ no se generó, copiame el error de arriba')

## Celda 3B — Solo si la Celda 3 dio `CUDA out of memory`
Baja la resolución a 192 (menos VRAM). Si aun así falla: `Entorno de ejecución → Reiniciar sesión` y corré esta celda directo (Celda 2 primero para tener `IMG`).

In [ ]:
import os, torch
os.chdir('/content/Hunyuan3D-2')
from PIL import Image
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
img = Image.open(IMG).convert('RGBA')
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=192,
            generator=torch.manual_seed(0))[0]
OUT = '/content/hunyuan_mesh.glb'
mesh.export(OUT)
print('✅', OUT, round(os.path.getsize(OUT)/1024,1), 'KB' if os.path.exists(OUT) else 'no se generó')

## Celda 3C (OPCIONAL) — Color y textura automática 🎨
**Solo para probar.** Le pinta color/textura a la malla. Es **pesado** en la T4 y puede fallar (compilación de módulos CUDA o memoria). El `.glb` de forma (Celda 3) ya lo tenés guardado igual.

**Para que NO se quede sin memoria mientras texturiza (recomendado):**
1. Corré primero la **Celda 3C-1** (compila los módulos de textura — una sola vez, queda en disco).
2. Después: `Entorno de ejecución → Reiniciar sesión`.
3. Corré la **Celda 2** (para tener `IMG`) y luego la **Celda 3C-2**. Al arrancar en sesión fresca, el modelo de forma (~10 GB) ya no ocupa VRAM → entra la textura.

In [ ]:
# Celda 3C-1 — compilar los modulos CUDA de textura (una vez; queda instalado en disco)
import os
os.chdir('/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer')
!python3 setup.py install 2>&1 | tail -3
os.chdir('/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer')
!python3 setup.py install 2>&1 | tail -3
os.chdir('/content/Hunyuan3D-2')
print('✅ Compilado (si no hubo error rojo). Recomendado: Reiniciar sesión, correr Celda 2, y luego la 3C-2.')

In [ ]:
# Celda 3C-2 — texturizar (con ahorros de memoria para la T4)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # menos fragmentacion -> menos OOM
import gc, torch, trimesh
os.chdir('/content/Hunyuan3D-2')
from PIL import Image

gc.collect(); torch.cuda.empty_cache()

# Cargar la malla desde el .glb de la Celda 3 (no depende de tener el modelo de forma en memoria)
mesh = trimesh.load('/content/hunyuan_mesh.glb', force='mesh')

# Menos caras = menos memoria al texturizar (y archivo mas liviano)
try:
    from hy3dgen.shapegen import FaceReducer
    mesh = FaceReducer()(mesh, max_facenum=40000)
    print('malla reducida a <=40k caras')
except Exception as e:
    print('no se pudo reducir la malla:', e)

# PARCHE trust_remote_code: el pipeline de pintura carga por dentro el modelo
# 'hunyuanpaint' (codigo custom) y diffusers nuevo exige trust_remote_code=True,
# que el codigo interno de Hunyuan no pasa -> lo forzamos globalmente.
import diffusers
from diffusers import DiffusionPipeline
if not getattr(DiffusionPipeline.from_pretrained, '_trc_patched', False):
    _orig_fp = DiffusionPipeline.from_pretrained.__func__
    def _fp_trc(cls, *a, **k):
        k.setdefault('trust_remote_code', True)
        return _orig_fp(cls, *a, **k)
    _fp_trc._trc_patched = True
    DiffusionPipeline.from_pretrained = classmethod(_fp_trc)
    print('parche trust_remote_code aplicado')

# Pipeline de pintura, con offload a CPU si esta disponible (ahorra VRAM)
from hy3dgen.texgen import Hunyuan3DPaintPipeline
paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
for _m in ('enable_model_cpu_offload', 'enable_sequential_cpu_offload'):
    try:
        getattr(paint, _m)(); print('offload activado:', _m); break
    except Exception:
        pass

try:
    mesh_tex = paint(mesh, image=Image.open(IMG).convert('RGB'))
    OUT = '/content/hunyuan_textured.glb'
    mesh_tex.export(OUT)
    print('✅', OUT, round(os.path.getsize(OUT)/1024, 1), 'KB')
except Exception as e:
    torch.cuda.empty_cache()
    if 'out of memory' in str(e).lower():
        print('❌ Sin memoria (la T4 va muy justa para textura). Hacé: Reiniciar sesión → Celda 2 (para IMG) → esta celda de nuevo (arranca sin el modelo de forma ocupando VRAM). Si aun asi falla, el color necesita una GPU mas grande.')
    else:
        raise

## Celda 4 — Descargar el `.glb`
Baja el **texturizado** si lo generaste; si no, el de forma.

In [ ]:
from google.colab import files
import os
OUT = '/content/hunyuan_textured.glb' if os.path.exists('/content/hunyuan_textured.glb') else '/content/hunyuan_mesh.glb'
print('Descargando:', OUT)
files.download(OUT)

---
### Resumen de parches
- **`AttributeError _blas_supports_fpe`** → Celda 1B (`numpy>=2.1`) + **Reiniciar sesión**. El error más común: pasa si te saltás el reinicio.
- **Celda 2 no sube nada / celular** → subí por el panel **Archivos** 📁 y corré la Celda 2 (detecta la imagen sola).
- **`Model path not exists` / baja 0 files** → estás con el `mini`. Usá `tencent/Hunyuan3D-2` (Celda 3).
- **Sale deforme / con fondo pegado** → subiste la imagen con fondo. Recortala (PNG transparente) y re-subí.
- **`CUDA out of memory`** → Celda 3B (octree 192) o reiniciar sesión.
- **Textura: `ValueError … trust_remote_code=True`** → ya está parcheado en la **Celda 1** (edita `multiview_utils.py` y le agrega `trust_remote_code=True`). Si clonaste el repo antes de este parche: corré `!sed -i 's/torch_dtype=torch.float16)/torch_dtype=torch.float16, trust_remote_code=True)/' hy3dgen/texgen/utils/multiview_utils.py` y **Reiniciá sesión**.
- La forma (Celda 3) siempre sale; el **color (Celda 3C)** es opcional y va justo de memoria en la T4.
- Cuando tengas el `.glb`, pasámelo y le pongo las animaciones mocap. Cualquier error rojo, copiámelo. 🐉